# FEMA Evidence & Oversight Graph — Judge Walkthrough (start here)

**Phebe Arouni & Tom Jacobson | Dataset Access track | Submission of October 29, 2026**

### In one sentence
A read-only **MCP server** with five tools that turns public FEMA Public Assistance project records into **review indicators** — each one shipped with its source records, its calculation, and its limits — so a human analyst can decide what deserves a second look.

### How to use this notebook (about 2-3 minutes)
1. Click **Run all** at the top (serverless compute is fine; no setup needed).
2. Read top to bottom. Every step has a short **"What this shows"** note, then a cell that proves it.
3. The last code cell prints a **Scorecard** comparing this live re-run with the officially recorded result.

**Safe to run:** it only writes to `/tmp`, calls the public OpenFEMA API, uses no secrets or tokens, and does not modify any table, job, or app.

### Roadmap
| Step | What you will see | Why it matters |
|---|---|---|
| 1 | The problem and what was built | Context |
| 2 | Install the exact pinned dependencies | Reproducibility |
| 3 | Unpack the **exact submitted zip** and verify its fingerprint | Proves you are judging the tested code |
| 4 | Run the automated tests (71 in the November 2 packet) and the linter | Code quality |
| 5 | Connect to the MCP server and list its tools | What an AI agent sees |
| 6 | Two live analyst scenarios on real FEMA data | The product working |
| 7 | Guardrails: bad inputs are rejected clearly | Safety |
| 8 | Tool results vs. an **independent SQL calculation** | Correctness |
| 9 | Recorded AI-model evidence (tool selection, no-tool baseline, hosted app) | Agent behavior |
| 10 | Scorecard | One-glance summary |
| 11 | What we do **not** claim, and where everything lives | Honesty and navigation |

## Step 1 — The problem and what was built

**Problem.** Public disaster-assistance data shows where money was obligated. An oversight analyst also needs to know *what deserves a second look*, *which records support the question*, and *what the data cannot prove*. An unexplained risk score, or an AI answer that is not tied to records, cannot be reviewed.

**Solution.** Read-only MCP tools over the public [OpenFEMA Public Assistance Funded Projects Details v2](https://www.fema.gov/openfema-data-page/public-assistance-funded-projects-details-v2) API:

| # | Tool | Analyst question it answers | Built-in limit it states |
|---|---|---|---|
| 1 | `find_repeated_worksheets` | Same disaster + applicant + title + category appearing more than once? | Matching titles are not proof of duplicate scope |
| 2 | `detect_category_outliers` | Which project amounts are far above their category's average? | Baseline is the retrieved subset, not matched engineering costs |
| 3 | `explain_risk_indicator` | How exactly is an indicator calculated? | Method card, not a policy decision |
| 4 | `generate_evidence_brief` | Package one project's source records for review | Does not re-decide the flag |

Also exposed: a provenance resource (`fema-pa://provenance`) and a review prompt (`fema_oversight_review`).

**How it flows**
```
Analyst question
      |
      v
AI model picks a tool (bounded loop, read-only tools only)
      |
      v
MCP tool  ---->  public OpenFEMA API (live records)
      |
      v
Indicators + source records + method + limits + provenance
      |
      v
Human analyst reviews  ---->  evidence brief
```

**Where it runs:** FastMCP server deployed as the authenticated Databricks App `2026h` (MCP endpoint `/mcp`). In this notebook we run the **same server in-process**, so no login or token is needed.

## Step 2 — Install the exact pinned dependencies

**What this shows:** the same library versions used for the recorded October 29 run. A pip message about `mlflow-skinny` / `starlette` may appear; it is harmless for this notebook.

In [0]:
%pip install -q fastmcp==4.0.11 pytest==9.1.1 pytest-asyncio==1.4.0 ruff==0.16.10

In [0]:
dbutils.library.restartPython()

## Step 3 — Unpack the exact submitted zip and verify its fingerprint

**What this shows:** we unpack the submission zip into `/tmp` and compute a SHA-256 fingerprint of all Python source files. If it equals the fingerprint saved by the recorded evaluation run inside the same zip, then **the code you are judging is byte-for-byte the code that was tested**. The November 2 zip carries its own recorded run (`eval/results/20261102T152007Z/`); the October 29 zip carries `eval/results/20261029T111811Z/`.

**Which zip:** the cell uses `FEMA_Hackathon_Submission_2026-11-02.zip`, the November 2 packet with the redesigned app page, deck, video and judge documents. It looks first in this folder, then in the volume `/Volumes/users/phebe_arouni/hackathon_submissions/20261102/`. If neither copy can be read, it falls back to the official `FEMA_Hackathon_Submission_2026-10-29.zip` in this folder. The cell prints which file and which recorded run it used. The two zips have different Python code: on November 2 the repeated-worksheet rule changed to compare titles while ignoring capitals, punctuation and numbers. Each zip is checked against its own recorded run.

In [0]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path

# Use the November 2 packet (new app page, deck, video and judge documents; same Python code).
# The first readable copy wins. If neither copy of it can be read, fall back to the official October 29 zip.
CANDIDATE_ZIPS = [
    "/Workspace/Users/phebe.arouni@hq.dhs.gov/hackathon_submissions/20261029/FEMA_Hackathon_Submission_2026-11-02.zip",
    "/Volumes/users/phebe_arouni/hackathon_submissions/20261102/FEMA_Hackathon_Submission_2026-11-02.zip",
    "/Workspace/Users/phebe.arouni@hq.dhs.gov/hackathon_submissions/20261029/FEMA_Hackathon_Submission_2026-10-29.zip",
]
def _readable(path):
    try:
        with open(path, "rb") as fh:
            return fh.read(4) == b"PK\x03\x04"
    except OSError:
        return False
SUBMISSION_ZIP = next(p for p in CANDIDATE_ZIPS if _readable(p))
print("Using:", SUBMISSION_ZIP)
print("Zip SHA-256:", hashlib.sha256(open(SUBMISSION_ZIP, "rb").read()).hexdigest())
WORK = Path("/tmp/judge_walkthrough")
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(SUBMISSION_ZIP).extractall(WORK)
ROOT = WORK / "fema-evidence-oversight"
# The recorded evaluation run shipped in this zip: the November 2 run if present, else the October 29 run.
RUNS = ["20261102T152007Z", "20261102T120900Z", "20261029T111811Z"]
EVIDENCE = next(ROOT / "eval/results" / r for r in RUNS if (ROOT / "eval/results" / r / "summary.json").exists())
print("Recorded run:", EVIDENCE.name)
sys.path.insert(0, str(ROOT / "src"))

# The scorecard is filled in as we go: check -> (recorded run, re-run now, pass?)
SCORE = {}

# Fingerprint of the source code, computed exactly as the evaluation harness does
h = hashlib.sha256()
for f in sorted((ROOT / "src").rglob("*.py")):
    h.update(f.relative_to(ROOT).as_posix().encode())
    h.update(f.read_bytes())
recorded = json.loads((EVIDENCE / "summary.json").read_text())
SCORE["Source fingerprint matches tested code"] = (
    recorded["source_sha256"][:16] + "...", h.hexdigest()[:16] + "...", h.hexdigest() == recorded["source_sha256"])

print("Submission contents (top level):")
for p in sorted(ROOT.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))
print("\nRecorded fingerprint :", recorded["source_sha256"])
print("Fingerprint right now:", h.hexdigest())
print("MATCH" if h.hexdigest() == recorded["source_sha256"] else "MISMATCH - code differs from the tested release")

## Step 4 — Run the automated tests and the linter

**What this shows:** automated tests (71 in the November 2 packet, 64 in the October 29 zip) covering input validation, HTTP security boundaries, tool registration and read-only annotations, project identity, the review rules, grounded summaries, and the browser routes. The linter (`ruff`) checks code style. Tests use no network.

In [0]:
import re, subprocess

env = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONPYCACHEPREFIX": "/tmp/pycache"}
tests = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "--color=no"],
                       cwd=ROOT, env=env, capture_output=True, text=True)
lint = subprocess.run([sys.executable, "-m", "ruff", "check", ".", "--no-cache"],
                      cwd=ROOT, capture_output=True, text=True)

last_line = tests.stdout.strip().splitlines()[-1]
passed = int(re.search(r"(\d+) passed", last_line).group(1)) if "passed" in last_line else 0
failed = int(re.search(r"(\d+) failed", last_line).group(1)) if "failed" in last_line else 0
print("pytest:", last_line)
print("ruff  :", lint.stdout.strip() or lint.stderr.strip())

m = recorded["metrics"]
SCORE["Automated tests passing"] = (f"{m['pytest_passed']}/{m['pytest_cases']}", f"{passed}/{passed + failed}",
                                    tests.returncode == 0 and passed == m["pytest_cases"])
SCORE["Lint (ruff) clean"] = ("pass", "pass" if lint.returncode == 0 else "fail", lint.returncode == 0)

## Step 5 — Connect to the MCP server and see what an AI agent sees

**What this shows:** we start the real server object in-process and connect with a standard MCP client — the same protocol the deployed Databricks App speaks at `/mcp`. Every tool is marked **read-only** and **non-destructive**.

In [0]:
import pandas as pd
from fastmcp import Client
from fema_oversight.app import mcp

async def call(tool, args):
    """Call one MCP tool through the MCP protocol and return its JSON result."""
    async with Client(mcp) as c:
        return (await c.call_tool(tool, args)).data

async with Client(mcp) as c:
    tools = await c.list_tools()
    prompts = await c.list_prompts()
    resources = await c.list_resources()

tool_table = pd.DataFrame([{
    "tool": t.name,
    "read_only": t.annotations.read_only_hint,
    "destructive": t.annotations.destructive_hint,
    "inputs": ", ".join(t.input_schema.get("properties", {}).keys()),
    "description (first line)": (t.description or "").strip().splitlines()[0],
} for t in tools])
display(tool_table)
print("Prompts  :", [p.name for p in prompts])
print("Resources:", [str(r.uri) for r in resources])

tool_names = [t.name for t in tools]
SCORE["MCP tools registered (all read-only)"] = (
    str(len(recorded["tools"])), f"{len(tool_names)} ({sum(bool(t.annotations.read_only_hint) for t in tools)} read-only)",
    tool_names == recorded["tools"] and all(t.annotations.read_only_hint for t in tools))

## Step 6 — Two live analyst scenarios on real FEMA data

These cells call the tools exactly as an AI agent would, against the **live** public OpenFEMA API. Counts can change slightly if FEMA updates records; Step 8 handles that by checking tool and control on the same data.

**What to look for in every answer:** each one carries `data_scope` (how many records were retrieved and whether a cap was hit), a `caveat` ("not a finding"), and `_provenance` (source, method, limits).

### Scenario A — Category cost outliers, Florida disaster 4673 (Hurricane Ian), Category C (Roads and Bridges)
**Analyst asks:** *"Which road and bridge projects are 3+ standard deviations above the category average, and how is that calculated? Package one for review."*

**What to look for:** the category mean, standard deviation and record count are returned with each answer, so the analyst can check the arithmetic. The explanation tool returns the formula, thresholds and limits. The top flagged project is then packaged as an **evidence brief** (full identity: disaster + applicant + worksheet) for a human reviewer.

In [0]:
co = await call("detect_category_outliers", {"damage_category_code": "C", "state_code": "FL", "disaster_number": 4673,
                                             "pagination": {"limit": 10, "offset": 0}})
print(f"Outliers: {co['total']}   Category baseline: {co['category_stats']}")
print("Data scope:", co["data_scope"])
display(pd.DataFrame(co["items"]))

ex = await call("explain_risk_indicator", {"indicator": "category_outlier"})
print("\n--- How this indicator works ---")
for k in ["name", "calculation", "thresholds", "limitations"]:
    print(f"{k:12}: {ex[k]}")

# Package the top flagged project into an evidence brief (full identity: disaster + applicant + worksheet)
first = co["items"][0]
brief = await call("generate_evidence_brief", {"pw_number": first["pw_number"], "disaster_number": first["disaster_number"],
                                               "applicant_id": first["applicant_id"], "indicator": "category_outlier"})
print("\n--- Evidence brief ---")
display(pd.DataFrame(brief["source_records"]))
for k in ["rationale", "limitations", "review_status"]:
    print(f"{k:14}: {brief[k]}")

### Scenario B — Repeated worksheets, North Carolina disaster 4827 (Tropical Storm Helene, 2024)
**Analyst asks:** *"Does the same applicant have several worksheets with matching titles for the same type of work?"*

**What to look for:** titles are compared after ignoring capitals, punctuation and standalone numbers, so "Culverts Small Project #1" and "Culverts Small Project # 4" match. FEMA titles are almost never repeated exactly (the October 29 exact-title rule found **zero** groups here). This is the same disaster as the live app's "Is the same project listed more than once?" card.

The cell labels how close each group is and lists the closest first, the same way the live app does:
* **1 same wording:** same words and numbers; only capitals, spacing or dashes differ. Here: two large state road-repair worksheets, "Div 13 Yancey" and "DIV 13 - Yancey" ($1.38 million and $1.64 million estimated).
* **2 numbered series:** only one number changes, such as Site 1 and Site 2. Often legitimate (separate buildings, sites or phases).
* **3 several differences:** several numbers or symbols differ (street numbers, ranges, < and >), so these are more likely separate sites.

It then shows every worksheet in the closest group side by side, read directly from OpenFEMA. With the October 29 zip this cell still shows zero.

In [0]:
# Scenario B: repeated worksheets for the same applicant, title and category
# North Carolina disaster 4827 (Tropical Storm Helene, 2024), the same example as the live app's card.
import re
from fema_oversight.utils import fetch_pa_records

rw = await call("find_repeated_worksheets", {"state_code": "NC", "disaster_number": 4827,
                                             "pagination": {"limit": 100, "offset": 0}})
print(f"Repeated groups: {rw['total']}")
print("Data scope     :", rw["data_scope"])
print("Caveat         :", rw["caveat"])

# How close is each group? The live app labels groups the same way and lists the closest first:
#   1 same wording        - same words and numbers; only capitals, spacing or dashes differ
#   2 numbered series     - only one number changes (Site 1, Site 2)
#   3 several differences - several numbers or symbols differ (street numbers, ranges, < >)
_CONNECT = re.compile(r"[\s\-\u2013\u2014,.:;_/()'\"#]+")
def closeness(titles):
    words = [re.findall(r"[a-z0-9]+", str(t).lower()) for t in titles]
    syms = [re.sub(r"[A-Za-z0-9]", "", _CONNECT.sub("", str(t))) for t in titles]
    same_syms = all(s == syms[0] for s in syms)
    if same_syms and all(w == words[0] for w in words):
        return "1 same wording"
    if same_syms and all(len(w) == len(words[0]) for w in words):
        vary = [i for i in range(len(words[0])) if any(w[i] != words[0][i] for w in words)]
        if len(vary) == 1 and all(w[vary[0]].isdigit() for w in words):
            return "2 numbered series"
    return "3 several differences"

groups = []
for g in rw["items"]:
    titles = ([w["application_title"] for w in g["worksheets"]] if g.get("worksheets")
              else g.get("application_titles") or [g["application_title"]])
    groups.append({**g, "closeness": closeness(titles)})
groups.sort(key=lambda g: (g["closeness"], -g["worksheet_count"], -g["total_federal_share_obligated"]))
if groups:
    print("\nCloseness of the", len(groups), "groups returned:",
          pd.Series([g["closeness"] for g in groups]).value_counts().sort_index().to_dict())
    cols = ["closeness", "applicant_id", "application_titles", "worksheet_count", "pw_numbers",
            "damage_category_description", "total_federal_share_obligated"]
    display(pd.DataFrame(groups[:5])[[c for c in cols if c in groups[0]]])

    # Side by side: every worksheet in the closest group, straight from OpenFEMA
    from fema_oversight.tools.repeated_worksheets import title_pattern
    top = groups[0]
    recs = await fetch_pa_records(
        odata_filter=(f"disasterNumber eq {int(top['disaster_number'])} and applicantId eq '{top['applicant_id']}'"
                      f" and damageCategoryCode eq '{top['damage_category_code']}'"),
        select=["pwNumber", "applicationTitle", "county", "projectSize", "projectAmount", "federalShareObligated"],
        top=10000)
    side = pd.DataFrame([r for r in recs if title_pattern(r.get("applicationTitle")) == top["title_pattern"]])
    print(f"\nClosest group, side by side ({top['closeness']}; applicant {top['applicant_id']}):")
    display(side.sort_values("pwNumber").reset_index(drop=True))
rule = await call("explain_risk_indicator", {"indicator": "repeated_worksheet"})
print("Grouping rule  :", rule["calculation"])
print("Limits         :", rule["limitations"])

## Step 7 — Guardrails: bad or risky inputs are rejected clearly

**What this shows:** invalid categories, malformed states, impossible bands and ambiguous project identities return a clear error and a hint — they do not silently run a different query. (These errors are returned as data, so an AI agent can read and correct them.)

In [0]:
bad_inputs = [
    ("Category 'Z' does not exist",       "detect_category_outliers", {"damage_category_code": "Z"}),
    ("State must be two letters",         "detect_category_outliers", {"damage_category_code": "C", "state_code": "Florida123"}),
    ("Repeat count below 2",              "find_repeated_worksheets", {"state_code": "FL", "min_repeat_count": 1}),
    ("Worksheet without disaster/applicant", "generate_evidence_brief", {"pw_number": 4902, "indicator": "category_outlier"}),
]
rows = []
for label, tool, args in bad_inputs:
    try:
        r = await call(tool, args)
        msg = r.get("error") if isinstance(r, dict) else None
        rejected = msg is not None
    except Exception as e:          # schema-level rejection by the MCP layer also counts as rejected
        msg, rejected = str(e).splitlines()[0][:160], True
    rows.append({"case": label, "tool": tool, "rejected": rejected, "message": msg})
guard = pd.DataFrame(rows)
display(guard)
SCORE["Invalid inputs rejected (this notebook)"] = ("n/a", f"{int(guard.rejected.sum())}/{len(guard)}", bool(guard.rejected.all()))

## Step 8 — Correctness: tool results vs. an independent SQL calculation

**What this shows:** for each detection tool, we take the input records, compute the expected answer **separately in SQLite** (plain SQL, not the tool's Python code), and check that the tool returns **exactly the same project identities**.

We do this twice:
* **8a — Archived inputs:** the exact records saved by the recorded run in this zip. Fully deterministic; must reproduce the recorded result.
* **8b — Fresh live inputs:** records pulled from OpenFEMA right now. Proves the check is not tied to one saved file.
* **8c — Seeded duplicate:** we copy one real record that is not yet in any group under a new worksheet number, and confirm both the tool and the SQL control now find exactly **one more** group.

For the repeat rule, the SQL control computes the title pattern with its own character-by-character function (no regular expressions), so it does not share the tool's code.

The independent SQL is printed first so you can read it.

In [0]:
import importlib.util, inspect
from unittest.mock import AsyncMock, patch
from fema_oversight.tools import category_outliers, repeated_worksheets
from fema_oversight.utils import fetch_pa_records

# Load the submission's own evaluation harness (eval/run_hackathon.py) for its control() and collect() helpers
spec = importlib.util.spec_from_file_location("run_hackathon", ROOT / "eval/run_hackathon.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)
print(inspect.getsource(harness.control))

In [0]:
CASES = [  # (rule, tool, module, tool arguments, OpenFEMA filter, record cap)
    ("repeated", "find_repeated_worksheets", repeated_worksheets, {"state_code": "FL", "disaster_number": 4673},
     "stateAbbreviation eq 'FL' and disasterNumber eq 4673", 10000),
    ("category", "detect_category_outliers", category_outliers,
     {"state_code": "FL", "damage_category_code": "C", "disaster_number": 4673},
     "stateAbbreviation eq 'FL' and disasterNumber eq 4673 and damageCategoryCode eq 'C'", 10000),
]

def tool_identities(rule, items, rows):
    """Turn tool output into the same tuple shape the SQL control returns (same logic as the harness)."""
    if rule != "repeated":
        return sorted([(r["disaster_number"], r["applicant_id"], r["pw_number"]) for r in items], key=str)
    if items and "title_pattern" in items[0]:   # November 2 rule: titles compared as patterns
        return sorted([(r["disaster_number"], r["applicant_id"], r["title_pattern"], r["damage_category_code"],
                        r["worksheet_count"], r["total_federal_share_obligated"]) for r in items], key=str)
    code = {x.get("damageCategoryDescrip"): x.get("damageCategoryCode") for x in rows}
    return sorted([(r["disaster_number"], r["applicant_id"], r["application_title"], code.get(r["damage_category_description"]),
                    r["worksheet_count"], r["total_federal_share_obligated"]) for r in items], key=str)

async def compare(rows, rule, tool, module, args):
    expected = [tuple(x) for x in harness.control(rows, rule)]
    with patch.object(module, "fetch_pa_records", AsyncMock(return_value=rows)):  # feed the tool the same records
        async with Client(mcp) as c:
            items, _ = await harness.collect(c, tool, args)
    actual = [tuple(x) for x in tool_identities(rule, items, rows)]
    return len(rows), len(actual), len(expected), actual == expected

recorded_checks = {c["name"]: c for c in recorded["checks"]}
results = []
for rule, tool, module, args, flt, cap in CASES:
    archived_rows = json.loads((EVIDENCE / f"snapshot_{rule}.json").read_text())
    n, t, k, ok = await compare(archived_rows, rule, tool, module, args)
    results.append({"check": "8a archived", "tool": tool, "input_records": n, "tool_count": t, "sql_control_count": k, "exact_match": ok})
    rec = recorded_checks[f"{rule}_sqlite_control"]
    SCORE[f"Tool = SQL control, archived ({rule})"] = (f"{rec['tool_count']} = {rec['control_count']}", f"{t} = {k}",
                                                       ok and t == rec["tool_count"])

    live_rows = await fetch_pa_records(odata_filter=flt, top=cap)
    n, t, k, ok = await compare(live_rows, rule, tool, module, args)
    results.append({"check": "8b live now", "tool": tool, "input_records": n, "tool_count": t, "sql_control_count": k, "exact_match": ok})
    SCORE[f"Tool = SQL control, live now ({rule})"] = ("n/a", f"{t} = {k}", ok)

# 8c: seed one duplicate of a record that is not yet in any group
from collections import Counter
rows = json.loads((EVIDENCE / "snapshot_repeated.json").read_text())
repeat_args = {"state_code": "FL", "disaster_number": 4673}
_, base_t, base_k, base_ok = await compare(rows, "repeated", "find_repeated_worksheets", repeated_worksheets, repeat_args)
pattern = getattr(repeated_worksheets, "title_pattern", lambda t: t or "")   # October 29 code compares exact titles
gkey = lambda r: (r.get("applicantId"), pattern(r.get("applicationTitle")), r.get("damageCategoryCode"))
sizes = Counter(gkey(r) for r in rows)
original = next(r for r in rows if all(gkey(r)) and sizes[gkey(r)] == 1)
seeded = {**original, "pwNumber": 999999}   # same disaster/applicant/title/category, new worksheet number
n, t, k, ok = await compare(rows + [seeded], "repeated", "find_repeated_worksheets", repeated_worksheets, repeat_args)
results.append({"check": "8c seeded duplicate", "tool": "find_repeated_worksheets", "input_records": n,
                "tool_count": t, "sql_control_count": k, "exact_match": ok})
SCORE["Seeded duplicate found by tool and control"] = ("n/a", f"{base_t} -> {t} groups (control {base_k} -> {k})",
                                                       ok and base_ok and t == base_t + 1)
print(f"Seeded duplicate of: '{original['applicationTitle']}' (applicant {original['applicantId']})")

display(pd.DataFrame(results))

## Step 9 — Recorded AI-model evidence (read from the saved run in this zip)

These checks used a hosted model (`databricks-claude-haiku-4-5`, temperature 0) and the deployed app, so they are **read from the saved evidence** rather than re-run (re-running needs model-serving access and costs money). All raw responses are in the recorded run folder inside the zip (`eval/results/20261102T152007Z/` for the November 2 packet, `eval/results/20261029T111811Z/` for October 29).

This walkthrough shows the recorded results for the two checks it covers (category cost outliers and repeated worksheets). Counts below are computed from those rows.

* **9a Tool selection:** analyst questions — did the model pick the right first tool?
* **9b No-tool baseline:** the same model answering exact-count questions *without* the tools.
* **9c Grounded workflows:** end-to-end review conversations; the displayed summary is assembled from tool evidence, not free model prose.
* **9d Hosted app smoke test:** the authenticated Databricks App answered health and MCP calls.

In [0]:
import re
# Show only the recorded rows for the checks covered in this walkthrough
OUT_OF_SCOPE = re.compile(r"cost.?share|federal.?share|4734", re.I)
def in_scope_question(x):
    return x.get("expected_tool") != "get_cost_share_anomalies" and not OUT_OF_SCOPE.search(x.get("question", ""))
def in_scope_workflow(w):
    return not OUT_OF_SCOPE.search(w.get("question", "")) and not any(
        t["tool"] == "get_cost_share_anomalies" or OUT_OF_SCOPE.search(json.dumps(t.get("arguments"))) for t in w["tool_calls"])

sel = [x for x in json.loads((EVIDENCE / "model_tool_selection.json").read_text()) if in_scope_question(x)]
print("9a  Tool selection")
display(pd.DataFrame(sel)[["question", "expected_tool", "selected_tool", "pass"]])
SCORE["Model picks correct first tool (recorded)"] = (f"{sum(x['pass'] for x in sel)}/{len(sel)}", "not re-run", all(x["pass"] for x in sel))

base = json.loads((EVIDENCE / "baseline_count_comparison.json").read_text())
recs = [r for r in base["records"] if r["case"] != "cost_share" and not OUT_OF_SCOPE.search(r.get("question", ""))]
n = len(recs)
tool_ok = sum(bool(r["tool_exact_match"]) for r in recs)
no_tool_ok = sum(bool(r["no_tool_exact_match"]) for r in recs)
abstained = sum(bool(r["no_tool_abstained_or_invalid"]) for r in recs)
print("\n9b  Exact-count questions, with tools vs. without tools")
display(pd.DataFrame([{"case": r["case"], "repeat": r["repeat"], "expected": r["expected_count"], "with_tools": r["tool_count"],
                       "with_tools_exact": r["tool_exact_match"], "without_tools_answer": str(r["no_tool_response"])[:90] + "..."}
                      for r in recs]))
print(f"With tools: {tool_ok}/{n} exact.  Without tools: {no_tool_ok}/{n} exact, {abstained}/{n} abstained (no data access).")
SCORE["Exact counts with tools vs without (recorded)"] = (f"{tool_ok}/{n} vs {no_tool_ok}/{n}", "not re-run", n > 0 and tool_ok == n)

wf = [w for w in json.loads((EVIDENCE / "grounded_browser_workflows.json").read_text()) if in_scope_workflow(w)]
print("\n9c  End-to-end grounded review workflows")
display(pd.DataFrame([{"question": w["question"], "tools_called": ", ".join(t["tool"] for t in w["tool_calls"]),
                       "answer_mode": w["answer_mode"], "seconds": w["elapsed_seconds"], "pass": w["pass"]} for w in wf]))
print("Example displayed answer (first workflow, first 900 characters):\n")
print(wf[0]["answer"][:900])
SCORE["Grounded review workflows (recorded)"] = (f"{sum(w['pass'] for w in wf)}/{len(wf)}", "not re-run", all(w["pass"] for w in wf))

smoke_path = EVIDENCE / "hosted_smoke.json"
if smoke_path.exists():
    smoke = json.loads(smoke_path.read_text())
    print("\n9d  Hosted Databricks App smoke test:", json.dumps({k: smoke.get(k) for k in ["verified_at", "health_http", "remote_mcp_explanation_pass"]}))
    SCORE["Hosted app health + remote MCP call (recorded)"] = (f"HTTP {smoke.get('health_http')}, MCP pass={smoke.get('remote_mcp_explanation_pass')}", "not re-run",
                                                              smoke.get("health_http") == 200 and bool(smoke.get("remote_mcp_explanation_pass")))
else:
    print("\n9d  No hosted smoke record in this recorded run.")
    SCORE["Hosted app health + remote MCP call (recorded)"] = ("not recorded", "not re-run", False)

## Step 10 — Scorecard

**Recorded** = the recorded run shipped in the zip Step 3 used: the November 2 run (`eval/results/20261102T152007Z/summary.json`, MLflow run `ff6c41f62dc04e2c929fea1e08ba3e21`) or the official October 29 run (`eval/results/20261029T111811Z/summary.json`, MLflow run `7445e452abfc4cd89896658d4ab7f44a`). **This run** = what this notebook just measured. "not re-run" rows come from saved model/app evidence (Step 9).

In [0]:
card = pd.DataFrame([{"check": k, "recorded_" + EVIDENCE.name[:8]: v[0], "this_run": v[1], "pass": bool(v[2])} for k, v in SCORE.items()])
display(card)
print(f"\n{int(card['pass'].sum())}/{len(card)} checks pass.")
print("ALL CHECKS PASS" if card["pass"].all() else "SOME CHECKS FAILED - see rows with pass = False")

## Step 11 — What we do **not** claim

* **Not a fraud detector, eligibility engine or payments workflow.** Every output is a *review indicator*; a human decides.
* **Category baselines** use the retrieved subset (same category and filters), not engineering-matched peers.
* **Retrieval caps:** 10,000 records per check (repeats, categories). `data_scope.possibly_truncated` warns when a cap is hit.
* **Model prose can still be wrong.** An October 26 model answer added unsupported legal citations; that trace is kept. Since October 29 the displayed summary is assembled from tool evidence. Tool and filter choices still need human review.
* **Small evaluation sets** (shown here: 8 tool-selection questions, 10 exact-count questions, 2 workflows, 2 control comparisons). No general accuracy, zero-hallucination or audit-savings claim is made.
* Data comes from FEMA Grants Manager, not FEMA's official financial system.

## Where everything lives

| Item | Location |
|---|---|
| Submitted code + evidence (this folder) | `FEMA_Hackathon_Submission_2026-10-29.zip` (official October 29 submission) |
| November 2 packet (used by Step 3 when readable) | `FEMA_Hackathon_Submission_2026-11-02.zip`, in this folder or `/Volumes/users/phebe_arouni/hackathon_submissions/20261102/` |
| Inside the zip: overview | `SUBMISSION.md`, `README.md`, `docs/JUDGE_BRIEF.md` |
| Inside the zip: design / security / limits | `docs/TECHNICAL_DESIGN.md`, `SECURITY.md`, `docs/LIMITATIONS_AND_ROADMAP.md` |
| Inside the zip: evaluation write-up | `eval/evaluation_documentation.md` (and `.docx`) |
| Inside the zip: raw evidence of the recorded run | `eval/results/20261102T152007Z/` (November 2 packet) and `eval/results/20261029T111811Z/` (October 29) |
| Inside the zip: deck and video | `presentation/FEMA_Evidence_Oversight_Graph.pptx`, `..._walkthrough.mp4` |
| Public repository | https://github.com/phebeong/2026-HACKATHON (commit `f21fe97`) |
| Live Databricks App (workspace login required) | https://2026h-7474654188777746.aws.databricksapps.com — MCP at `/mcp`. Redesigned November 2, 2026, with the new repeated-worksheet rule. On November 3 it gained a side-by-side view of each repeated group, closest matches first, and the repeat card moved to North Carolina disaster 4827, so its code is now newer than the November 2 zip (see `README_FOR_JUDGES.md`, "About the live app version") |
| Saved validation job (on demand) | Job `67919837469456` |
| MLflow experiment / reference runs | `1305734463288268` / `ff6c41f62dc04e2c929fea1e08ba3e21` (November 2), `7445e452abfc4cd89896658d4ab7f44a` (October 29) |

**Thank you for reviewing.**